# FSD and FSD Cube: geometry, tiles, pixels and nominal readout

Run all cells. No GPU or larnd-sim Python installation is needed; the adjacent larnd-sim repository provides YAML configuration. Change `LARND_REPO` below (or set `LARND_SIM_REPO`) to use another checkout.

**Scope:** simulation readout bounds, not a mechanical CAD model. Pixel points mark centers, not pad sizes. Coordinates are edep-sim/geometry **x = drift, y = vertical, z = transverse**, in centimeters; the display places y vertically. larnd-sim internally swaps x and z. Layout TPC labels can start at 1; simulation pixel IDs encode a zero-based plane index.

Drag to rotate; scroll to zoom; hover for tile labels. The pixel map exposes nominal YAML addresses, not data-run routing aliases or channel masks.

In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts at the repository root or visualizations/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'visualizations/geometry.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from dune-nd-mip-calib or visualizations/')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
LARND_REPO = Path(os.environ.get('LARND_SIM_REPO', str(ROOT.parent / 'larnd-sim'))).expanduser()
OUT = ROOT / 'visualizations/outputs'
OUT.mkdir(exist_ok=True)
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
from IPython.display import display
from visualizations.geometry import load_geometry, geometry_figure, pixel_figure, tile_pixels, repositioning_boxes, export_html
print('Reading larnd-sim configuration from', LARND_REPO)


In [ ]:
fsd = load_geometry(LARND_REPO, 'fsd')
fsd_cube = load_geometry(LARND_REPO, 'fsd_cube')
for g in (fsd, fsd_cube):
    display(g.summary())
    print('TPC bounds [x, y, z], cm:', g.tpc_bounds)
comparison = geometry_figure(fsd, fsd_cube)
comparison.show()

## Pixel centers and readout addresses

The full detector view renders tile planes rather than hundreds of thousands of markers. The tile view shows every configured pixel. Select a chip to inspect its channels; hover for `(io_group, io_channel, chip_id, channel_id)` and the simulation pixel ID.

The saved HTML is interactive (pan, zoom, hover), but changing detector/chip settings requires the notebook kernel.

In [ ]:
tile_view = pixel_figure(fsd_cube, tile=1)
tile_view.show()
# A chip-level view without widgets:
# pixel_figure(fsd_cube, tile=1, chip=11, color_by='channel_id').show()

In [ ]:
import ipywidgets as widgets
options = [(f'{g.name} / tile {t}', (i,t)) for i,g in enumerate((fsd,fsd_cube)) for t in g.tile_centers]
choice = widgets.Dropdown(options=options, value=(1,1), description='Tile:')
chip_choice = widgets.Dropdown(options=[('All chips', None)] + [(str(c),c) for c in sorted(set(tile_pixels(fsd_cube,1)['chip_id']))], description='Chip:')
color_choice = widgets.Dropdown(options=['chip_id','channel_id','io_channel','io_group'], description='Color:')
active_pixel_view = {'figure': tile_view}
def draw_pixels(selection, chip, color):
    g = (fsd, fsd_cube)[selection[0]]
    fig = pixel_figure(g, selection[1], chip=chip, color_by=color)
    active_pixel_view['figure'] = fig
    fig.show()
controls = widgets.interactive_output(draw_pixels, dict(selection=choice, chip=chip_choice, color=color_choice))
display(widgets.HBox([choice,chip_choice,color_choice]), controls)

## Repositioning FSD deposits for FSD Cube

The original shower notebook scans **40 harvesting windows**. Choose a source window below. Orange is the source; purple is the translated destination; green is the actual FSD Cube readout volume from YAML. The harvesting window and readout boundaries are not identical. This is a coordinate overlay, **not a physical FSD Cube installed inside FSD**. Only translation is represented; no new particle transport is performed.

In [ ]:
active_translation = {}
def show_translation(index=25):
    source, destination, shift = repositioning_boxes(index)
    print('Translation [dx, dy, dz] in cm:', shift)
    actual = next(iter(fsd_cube.tpc_bounds.values()))
    print('Destination harvesting window minus active bounds [cm]:\n', destination-actual)
    fig = geometry_figure(fsd, opacity=.12, overlay=[
        (source, 'Source harvesting window', '#d97706'),
        (destination, 'Destination harvesting window', '#9333ea'),
        (actual, 'FSD Cube readout bounds (coordinate overlay)', '#059669')])
    active_translation['figure'] = fig
    fig.show()
    return fig
translation = show_translation(25)

In [ ]:
translation_control = widgets.interactive_output(lambda index: show_translation(index), {'index': (window := widgets.IntSlider(value=25,min=0,max=39,description='Window:',continuous_update=False))})
display(window, translation_control)

## Save figures

These HTML files include Plotly itself and can be opened offline. They preserve rotation, zoom and hover. Notebook widgets are not embedded. Export reflects the current pixel widget selection; the translation export is the example generated above.

In [ ]:
for fig, name in [(comparison,'detector_geometry.html'),(active_pixel_view['figure'],'pixel_layout.html'),(active_translation['figure'],'repositioning.html')]:
    print(export_html(fig, OUT/name))

## Sources and limits

- Geometry: `larndsim/config/config.yaml`, the selected detector-properties YAML, and pixel-layout YAML; the summary includes the layout SHA-256.
- Transform and routing conventions: `larndsim/consts/detector.py`, `larndsim/fee.py::rotate_tile` and `save_packets`, and the FSD mapping notebook.
- Harvesting windows: `2026_05_27.skim_more_showers.ipynb`, with its constants preserved independently from detector bounds.
- Mechanical cryostat, field-cage and support structures are not inferred from pixel YAML. The existing ND_Production ROOT geometry viewer can render GDML for those details.
- Disabled/noisy channels, per-run calibration, and alternate routes are not represented. This viewer deliberately rejects multi-module configurations.